# TFT Perfect-Forecast Experiment — Cáceres Solar Forecast

**Experiment:** all ERA5 weather features are reclassified as `time_varying_known_reals`,
simulating perfect weather forecasts available at prediction time. This is **not realistic
for production** — it establishes the **performance ceiling** to determine whether
obtaining real NWP forecast data (e.g., from ECMWF or Open-Meteo) is worth pursuing.

The only difference vs `4_tft_training.ipynb` is the feature classification:
weather features move from `OBSERVED` (encoder-only) to `KNOWN_FUTURE` (encoder + decoder).
Everything else — data, HP search space, training config — remains identical for fair comparison.

**Original model results (no weather in decoder):**
- Val: R²=0.8299, MAE=183.24 MWh, RMSE=343.89 MWh
- Test: R²=0.8284, MAE=152.00 MWh, RMSE=295.61 MWh, MASE=0.9499

**Pipeline steps:**

| # | Step | Purpose |
|---|---|---|
| 0 | Imports & setup | Libraries, GPU detection, random seed |
| 1 | Configuration | Sequence lengths, feature lists (**weather → known-future**) |
| 2 | Data loading | Load CSVs, add time_idx and group_id |
| 3 | TimeSeriesDataSet construction | All features as known-future, no unknown reals |
| 4 | Learning rate finder | Establish LR range before tuning |
| 5 | Hyperparameter tuning | Optuna TPE, 15 trials, same search space |
| 6 | Final model training | Best HPs, early stopping, checkpointing |
| 7 | Validation evaluation | Denormalized metrics (RMSE, MAE, R²) |
| 8 | Save artifacts | Model, study, metrics, predictions |


## Step 0 — Imports & setup

In [ ]:
import os, json, warnings, pickle
import numpy as np
import numpy._core.multiarray
import pandas as pd
import matplotlib.pyplot as plt

import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.tuner import Tuner

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss, MAE, RMSE
from pytorch_forecasting.data.encoders import TorchNormalizer

import optuna
from optuna.samplers import TPESampler

# Restore pre-PyTorch-2.6 torch.load behavior (weights_only=False by default)
# Needed because Lightning's LR finder and checkpoint restore store pytorch-forecasting
# objects that aren't in the safe globals allowlist. Safe here — we only load our own checkpoints.
import functools
_orig_torch_load = torch.load
@functools.wraps(_orig_torch_load)
def _patched_torch_load(*args, **kwargs):
    if kwargs.get('weights_only') is None:
        kwargs['weights_only'] = False
    return _orig_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pl.seed_everything(42)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
%pip install torchmetrics

## Step 1 — Configuration

In [ ]:
# ── Paths ──
DATA_DIR  = os.path.join(os.getcwd(), 'data')
MODEL_DIR = os.path.join(os.getcwd(), 'models', 'perfect_forecast')
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Sequence lengths ──
MAX_ENCODER_LENGTH    = 168   # 7 days of hourly history
MAX_PREDICTION_LENGTH = 24    # 1-day-ahead forecast

# ── Feature classification ──
# EXPERIMENT: all ERA5 weather features treated as known-future
# (simulates having perfect NWP forecasts available at prediction time)
KNOWN_FUTURE = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'doy_sin', 'doy_cos', 'solar_zenith', 'solar_azimuth', 'clearsky_ghi',
    # ERA5 weather — moved from OBSERVED to KNOWN_FUTURE for this experiment
    'dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa',
    'total_precip_mm', 'ssrd_wm2', 'strd_wm2', 'kt', 'dewpoint_depression_C',
]
OBSERVED = []  # all weather moved to known-future
TARGET = 'pv_generation_mwh'

# ── Denormalization parameters ──
with open(os.path.join(DATA_DIR, 'preprocessing_params.json')) as f:
    pp = json.load(f)
TARGET_MEAN = pp['target_mean']
TARGET_STD  = pp['target_std']

print(f"Encoder: {MAX_ENCODER_LENGTH}h ({MAX_ENCODER_LENGTH//24}d)")
print(f"Prediction: {MAX_PREDICTION_LENGTH}h")
print(f"Known future features: {len(KNOWN_FUTURE)} (includes weather!)")
print(f"Observed features: {len(OBSERVED)} (none — all weather is 'known')")
print(f"Target denorm: z * {TARGET_STD:.4f} + {TARGET_MEAN:.4f}")


## Step 2 — Load data & prepare for TimeSeriesDataSet

In [ ]:
# ── Load CSVs ──
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_processed.csv'),
                        parse_dates=['datetime_utc'], index_col='datetime_utc')
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'val_processed.csv'),
                        parse_dates=['datetime_utc'], index_col='datetime_utc')

print(f"Train: {len(train_df):,} rows  {train_df.index[0]} -> {train_df.index[-1]}")
print(f"Val:   {len(val_df):,} rows   {val_df.index[0]} -> {val_df.index[-1]}")

# ── Verify no NaN ──
assert train_df.isna().sum().sum() == 0, "NaN in training data"
assert val_df.isna().sum().sum() == 0, "NaN in validation data"

# ── Add time_idx: continuous integer across both splits ──
train_df = train_df.reset_index()
val_df   = val_df.reset_index()

train_df['time_idx'] = np.arange(len(train_df))
val_df['time_idx']   = np.arange(len(train_df), len(train_df) + len(val_df))

# ── Add constant group_id (single time series) ──
train_df['group_id'] = '0'
val_df['group_id']   = '0'

# ── Verify continuity ──
assert train_df['time_idx'].iloc[-1] + 1 == val_df['time_idx'].iloc[0], \
    "time_idx not continuous across train/val boundary"

print(f"\ntime_idx range — train: [0, {train_df['time_idx'].max()}], "
      f"val: [{val_df['time_idx'].min()}, {val_df['time_idx'].max()}]")
print(f"Columns: {list(train_df.columns)}")

## Step 3 — Construct TimeSeriesDataSet

The preprocessed data is already z-score standardized (train-set statistics applied to all splits).
`TimeSeriesDataSet` normally applies its own per-window normalization — we disable this by setting
`target_normalizer=None` and passing no scalers, to avoid double-normalization.

**Perfect-forecast experiment:** all features are `time_varying_known_reals`.
No `time_varying_unknown_reals` — the model sees weather for both past and future.


In [ ]:
# ── Training dataset ──
# Perfect-forecast experiment: no time_varying_unknown_reals.
# All features (including weather) are known into the future.
dataset_kwargs = dict(
    time_idx='time_idx',
    target=TARGET,
    group_ids=['group_id'],

    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    min_encoder_length=MAX_ENCODER_LENGTH,

    time_varying_known_reals=KNOWN_FUTURE,

    target_normalizer=None,
    scalers={col: None for col in KNOWN_FUTURE},

    add_relative_time_idx=True,
    add_encoder_length=True,
    allow_missing_timesteps=False,
)

# Only add time_varying_unknown_reals if OBSERVED is non-empty
if OBSERVED:
    dataset_kwargs['time_varying_unknown_reals'] = OBSERVED
    dataset_kwargs['scalers'].update({col: None for col in OBSERVED})

training_dataset = TimeSeriesDataSet(train_df, **dataset_kwargs)

print(f"Training dataset: {len(training_dataset)} samples")
print(f"Encoder length: {training_dataset.max_encoder_length}")
print(f"Prediction length: {training_dataset.max_prediction_length}")


In [ ]:
# ── Validation dataset ──
# Provide encoder context from the training tail.
# ignore_index=True gives a clean monotonic integer index — required because
# train_df.iloc[-168:] has index 17376-17543 and val_df has index 0-4343,
# so without it the concat index is non-monotonic and PyTorch Forecasting
# silently produces only 1 valid window instead of ~4321.
# min_prediction_idx is not needed: the context slice starts exactly
# MAX_ENCODER_LENGTH steps before val_df, so the first valid prediction
# window lands at val_df['time_idx'].min() automatically.
val_with_context = pd.concat([
    train_df.iloc[-MAX_ENCODER_LENGTH:],
    val_df
], ignore_index=True)

val_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    val_with_context,
    stop_randomization=True,
)

print(f"Validation dataset: {len(val_dataset)} samples")
assert len(val_dataset) > 100, \
    f"Val dataset too small ({len(val_dataset)}): check concat/context construction"

In [ ]:
# ── DataLoaders ──
BATCH_SIZE = 64

train_dataloader = training_dataset.to_dataloader(
    train=True, batch_size=BATCH_SIZE, num_workers=2
)
val_dataloader = val_dataset.to_dataloader(
    train=False, batch_size=BATCH_SIZE, num_workers=2
)

# Sanity check: inspect one batch
x, y = next(iter(train_dataloader))
print(f"Batch shapes:")
print(f"  Encoder cont. inputs: {x['encoder_cont'].shape}")
print(f"  Decoder cont. inputs: {x['decoder_cont'].shape}")
print(f"  Target:               {y[0].shape}")

## Step 4 — Learning rate finder

Run Lightning's LR finder to establish a reasonable learning rate range before HP tuning.
This prevents wasting Optuna trials on suboptimal LR regions.

In [ ]:
# Quick model with default HPs for LR finding
tft_tmp = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=0.001,
    hidden_size=32,
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=QuantileLoss(),
    optimizer='adam',
)

trainer_tmp = pl.Trainer(
    accelerator='auto',
    gradient_clip_val=0.1,
    max_epochs=3,
)

tuner = Tuner(trainer_tmp)
lr_finder = tuner.lr_find(
    tft_tmp,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
    min_lr=1e-6,
    max_lr=1.0,
    num_training=100,
)

fig = lr_finder.plot(suggest=True)
plt.title("Learning Rate Finder")
plt.show()

suggested_lr = lr_finder.suggestion()
print(f"Suggested LR: {suggested_lr:.6f}")

del tft_tmp, trainer_tmp
torch.cuda.empty_cache()

## Step 5 — Hyperparameter tuning (Optuna, TPE, 15 trials)

Same search space as the original model for fair comparison.
The key difference: all weather features are now available in the decoder.

| Hyperparameter | Search Space | Rationale |
|---|---|---|
| `hidden_size` | {16, 32, 64, 128} | Model capacity; 128 already large for 17k samples |
| `lstm_layers` | {1, 2} | Temporal processing depth |
| `attention_head_size` | {1, 2, 4} | Multi-head interpretable attention |
| `dropout` | Uniform(0.1, 0.4) | Primary regularization lever |
| `hidden_continuous_size` | {8, 16, 32} | Continuous variable processing width |
| `learning_rate` | LogUniform(1e-4, 1e-2) | Guided by LR finder |

Per-trial: max 10 epochs, early stopping patience 3 on val_loss.


In [ ]:
def optuna_objective(trial):
    """Single Optuna trial: sample HPs, train TFT, return val_loss."""

    # ── Sample hyperparameters ──
    hidden_size            = trial.suggest_categorical('hidden_size', [16, 32, 64, 128])
    lstm_layers            = trial.suggest_int('lstm_layers', 1, 2)
    attention_head_size    = trial.suggest_categorical('attention_head_size', [1, 2, 4])
    dropout                = trial.suggest_float('dropout', 0.1, 0.4)
    hidden_continuous_size = trial.suggest_categorical('hidden_continuous_size', [8, 16, 32])
    learning_rate          = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    # ── Create model ──
    tft = TemporalFusionTransformer.from_dataset(
        training_dataset,
        hidden_size=hidden_size,
        lstm_layers=lstm_layers,
        attention_head_size=attention_head_size,
        dropout=dropout,
        hidden_continuous_size=hidden_continuous_size,
        learning_rate=learning_rate,
        loss=QuantileLoss(),
        optimizer='adam',
        reduce_on_plateau_patience=3,
        output_size=7,
    )

    # ── Trainer (lightweight for tuning) ──
    trainer = pl.Trainer(
        max_epochs=10,  # 20 was gonna take way too long
        accelerator='auto',
        gradient_clip_val=0.1,
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=3, mode='min', verbose=False),
        ],
        enable_model_summary=False,
        enable_progress_bar=False,
        enable_checkpointing=False,
        logger=False,
    )

    try:
        trainer.fit(tft, train_dataloaders=train_dataloader,
                    val_dataloaders=val_dataloader)
        val_loss = trainer.callback_metrics['val_loss'].item()
    except Exception as e:
        print(f"Trial {trial.number} failed: {e}")
        val_loss = float('inf')
    finally:
        torch.cuda.empty_cache()

    return val_loss

In [ ]:
study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=42),
    study_name='tft_caceres_perfect_forecast_hp',
)

study.optimize(
    optuna_objective,
    n_trials=15,    # 25 was also unnecessarily long for this
    show_progress_bar=True,
    gc_after_trial=True,
)

print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best val_loss: {study.best_value:.6f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


In [ ]:
# ── Visualize the HP study ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
trial_numbers = [t.number for t in study.trials if t.value is not None and t.value < float('inf')]
trial_values  = [t.value for t in study.trials if t.value is not None and t.value < float('inf')]
best_so_far   = np.minimum.accumulate(trial_values)

axes[0].scatter(trial_numbers, trial_values, alpha=0.6, label='Trial value')
axes[0].plot(trial_numbers, best_so_far, 'r-', linewidth=2, label='Best so far')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Validation Loss')
axes[0].set_title('Optimization History')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Parameter importances (manual: use trial values)
param_names = list(study.best_params.keys())
importances = optuna.importance.get_param_importances(study)
axes[1].barh(list(importances.keys()), list(importances.values()))
axes[1].set_xlabel('Importance')
axes[1].set_title('Hyperparameter Importances')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6 — Train final model with best HPs

Full training with the best hyperparameters from the Optuna study.
Longer training budget (max 75 epochs) with patience 10 and model checkpointing.

In [ ]:
best = study.best_params

tft_final = TemporalFusionTransformer.from_dataset(
    training_dataset,
    hidden_size=best['hidden_size'],
    lstm_layers=best['lstm_layers'],
    attention_head_size=best['attention_head_size'],
    dropout=best['dropout'],
    hidden_continuous_size=best['hidden_continuous_size'],
    learning_rate=best['learning_rate'],
    loss=QuantileLoss(),
    optimizer='adam',
    reduce_on_plateau_patience=5,
    output_size=7,
)

print(f"Model parameters: {sum(p.numel() for p in tft_final.parameters()):,}")
print(f"Best HPs: {best}")

In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=MODEL_DIR,
    filename='tft_pf_best',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
)

trainer_final = pl.Trainer(
    max_epochs=75,
    accelerator='auto',
    gradient_clip_val=0.1,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10, mode='min', verbose=True),
        checkpoint_callback,
        LearningRateMonitor(logging_interval='epoch'),
    ],
    log_every_n_steps=50,
)

trainer_final.fit(
    tft_final,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

print(f"\nBest model saved to: {checkpoint_callback.best_model_path}")
print(f"Best val_loss: {checkpoint_callback.best_model_score:.6f}")


In [ ]:
# ── Plot training vs validation loss (generalization gap monitor) ──
train_losses = []
val_losses = []

# Extract from trainer's logged metrics
log_dir = trainer_final.logger.log_dir
metrics_file = os.path.join(log_dir, 'metrics.csv') if log_dir else None

if metrics_file and os.path.exists(metrics_file):
    metrics_df = pd.read_csv(metrics_file)
    # group by epoch and get last value per epoch
    if 'train_loss_epoch' in metrics_df.columns:
        train_loss_per_epoch = metrics_df.dropna(subset=['train_loss_epoch']).groupby('epoch')['train_loss_epoch'].last()
        val_loss_per_epoch = metrics_df.dropna(subset=['val_loss']).groupby('epoch')['val_loss'].last()

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(train_loss_per_epoch.index, train_loss_per_epoch.values, label='Train loss')
        ax.plot(val_loss_per_epoch.index, val_loss_per_epoch.values, label='Val loss')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Quantile Loss')
        ax.set_title('Training vs Validation Loss (generalization gap monitor)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.show()

        gap = val_loss_per_epoch.min() - train_loss_per_epoch.min()
        print(f"Generalization gap (val_min - train_min): {gap:.6f}")
    else:
        print("Training loss not logged per epoch. Check logger configuration.")
else:
    print("No metrics CSV found. Loss curves unavailable.")

## Step 7 — Validation evaluation

Generate predictions on the validation set and compute metrics on the original MWh scale.
This is the final validation — no further model changes after this point.

In [ ]:
def denormalize(z_values):
    """Convert z-score normalized values back to original MWh scale."""
    return z_values * TARGET_STD + TARGET_MEAN

def compute_metrics(actual_mwh, predicted_mwh):
    """Compute RMSE, MAE, R² on original MWh scale."""
    residuals = actual_mwh - predicted_mwh
    mae  = np.abs(residuals).mean()
    rmse = np.sqrt((residuals ** 2).mean())
    ss_res = (residuals ** 2).sum()
    ss_tot = ((actual_mwh - actual_mwh.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot
    return {'MAE_MWh': float(mae), 'RMSE_MWh': float(rmse), 'R2': float(r2)}

In [ ]:
# Load best checkpoint
best_model = TemporalFusionTransformer.load_from_checkpoint(
    checkpoint_callback.best_model_path
)
best_model.eval()

# Predict on validation set
val_predictions = best_model.predict(
    val_dataloader,
    mode='prediction',
    return_x=True,
    return_y=True,
)

val_raw = best_model.predict(
    val_dataloader,
    mode='raw',
    return_x=True,
)

In [ ]:
# Extract predictions and actuals
pred_z   = val_predictions.output.cpu().numpy()
actual_z = val_predictions.y[0].cpu().numpy()

# Flatten across all windows and horizons
pred_z_flat   = pred_z.flatten()
actual_z_flat = actual_z.flatten()

# Denormalize
pred_mwh   = denormalize(pred_z_flat)
actual_mwh = denormalize(actual_z_flat)

# Clip negative predictions to 0 (solar generation cannot be negative)
pred_mwh = np.clip(pred_mwh, 0, None)

val_metrics = compute_metrics(actual_mwh, pred_mwh)
print("Validation Metrics (original MWh scale):")
for k, v in val_metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"\nValidation quantile loss (normalized): {checkpoint_callback.best_model_score:.6f}")

In [ ]:
# ── Per-horizon metrics ──
horizon_metrics = []
for h in range(MAX_PREDICTION_LENGTH):
    pred_h   = denormalize(pred_z[:, h])
    actual_h = denormalize(actual_z[:, h])
    pred_h   = np.clip(pred_h, 0, None)
    m = compute_metrics(actual_h, pred_h)
    m['horizon_h'] = h + 1
    horizon_metrics.append(m)

horizon_df = pd.DataFrame(horizon_metrics)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['MAE_MWh', 'RMSE_MWh', 'R2']):
    ax.plot(horizon_df['horizon_h'], horizon_df[metric], marker='o', markersize=3)
    ax.set_xlabel('Forecast Horizon (hours)')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Forecast Horizon')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(horizon_df.to_string(index=False))

## Step 8 — Save artifacts

In [ ]:
# 1. Optuna study (for later analysis / warm-starting)
with open(os.path.join(MODEL_DIR, 'hp_study_pf.pkl'), 'wb') as f:
    pickle.dump(study, f)

# 2. Human-readable HP results
hp_results = {
    'best_trial': study.best_trial.number,
    'best_val_loss': study.best_value,
    'best_params': study.best_params,
    'n_trials': len(study.trials),
    'val_metrics_mwh': val_metrics,
    'sequence_config': {
        'max_encoder_length': MAX_ENCODER_LENGTH,
        'max_prediction_length': MAX_PREDICTION_LENGTH,
    },
    'training_config': {
        'batch_size': BATCH_SIZE,
        'max_epochs_final': 75,
        'early_stopping_patience': 10,
        'gradient_clip_val': 0.1,
        'optimizer': 'adam',
        'loss': 'QuantileLoss',
        'quantiles': [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98],
    },
}

with open(os.path.join(MODEL_DIR, 'hp_study_results_pf.json'), 'w') as f:
    json.dump(hp_results, f, indent=2)

# 3. Validation predictions
val_pred_df = pd.DataFrame({
    'actual_z': actual_z_flat,
    'predicted_z': pred_z_flat,
    'actual_mwh': actual_mwh,
    'predicted_mwh': pred_mwh,
})
val_pred_df.to_csv(os.path.join(MODEL_DIR, 'val_predictions_pf.csv'), index=False)

print("Artifacts saved:")
for f_name in sorted(os.listdir(MODEL_DIR)):
    f_path = os.path.join(MODEL_DIR, f_name)
    if os.path.isfile(f_path):
        size_mb = os.path.getsize(f_path) / 1e6
        print(f"  {f_name}: {size_mb:.1f} MB")


In [ ]:
print("=" * 70)
print("PERFECT-FORECAST EXPERIMENT COMPLETE")
print("=" * 70)
print(f"\nBest hyperparameters:")
for k, v in best.items():
    print(f"  {k}: {v}")
print(f"\nValidation metrics (MWh scale):")
for k, v in val_metrics.items():
    print(f"  {k}: {v:.4f}")

# ── Side-by-side comparison with original model ──
orig = {
    'MAE_MWh': 183.2382,
    'RMSE_MWh': 343.8855,
    'R2': 0.8299,
}
print(f"\n{'─' * 70}")
print(f"COMPARISON: Perfect Forecast vs Original (validation set)")
print(f"{'─' * 70}")
print(f"{'Metric':<15s} {'Original':>12s} {'Perfect FC':>12s} {'Change':>12s}")
print(f"{'─' * 52}")
for metric in ['MAE_MWh', 'RMSE_MWh', 'R2']:
    o = orig[metric]
    p = val_metrics[metric]
    if metric == 'R2':
        change = f"+{(p - o):.4f}" if p > o else f"{(p - o):.4f}"
        print(f"{metric:<15s} {o:12.4f} {p:12.4f} {change:>12s}")
    else:
        pct = (p - o) / o * 100
        print(f"{metric:<15s} {o:12.2f} {p:12.2f} {pct:+11.1f}%")

print(f"\nModel checkpoint: {checkpoint_callback.best_model_path}")
print(f"\nInterpretation:")
if val_metrics['R2'] > 0.90:
    print("  R² > 0.90 → Weather forecasts unlock substantial headroom.")
    print("  Recommended next step: obtain real NWP forecast data (e.g., Open-Meteo API).")
else:
    print("  R² improvement is modest → bottleneck may be elsewhere.")
    print("  Consider: data quality, model architecture, or aggregation approach.")
